# Laboratorul 5 — Stable Diffusion în practică

Notebook pentru Google Colab cu GPU (Runtime → Change runtime type → T4 GPU). Măsoară efectele setărilor asupra imaginilor, scorului CLIP și timpului. Rulările costisitoare sunt dezactivate implicit: setați **RUN_EXPERIMENTS = True** în celula de configurare numai după ce ați verificat GPU-ul, spațiul și accesul la model. Nu am executat descărcări sau experimente în acest notebook.

Păstrați prompturile sigure; nu generați persoane reale, logo-uri sau conținut ofensator. Rulați celulele în ordine.

In [ ]:
# Dependențe pentru Colab; folosim PyTorch-ul furnizat de runtime.
%pip -q install diffusers "transformers>=4.41,<5" accelerate torchmetrics matplotlib pandas

## Model, disponibilitate și licență

Modelul este încărcat explicit ca **stable-diffusion-v1-5/stable-diffusion-v1-5** în fp16, fără model alternativ automat. Repo-ul poate deveni indisponibil, mutat ori poate necesita acceptarea condițiilor/autentificare Hugging Face. Dacă încărcarea eșuează, notebook-ul oprește rularea și afișează cauza; verificați pagina modelului și solicitați profesorului o alternativă aprobată, apoi schimbați identificatorul explicit. Nu introduceți token-uri în celule sau output.

Citiți licența aplicabilă (CreativeML Open RAIL-M la distribuțiile uzuale SD 1.5) și condițiile publicate pe pagina modelului: respectați restricțiile de utilizare și distribuire. Reverificați licența și disponibilitatea la data rulării; nu este consultanță juridică.

In [ ]:
from __future__ import annotations
import copy, datetime as dt, json, math, platform, time, uuid
from dataclasses import dataclass
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image

RUN_EXPERIMENTS = True  # Setați True numai după ce GPU-ul/modelul sunt pregătite.
RUN_IMG2IMG = False
MODEL_ID = "stable-diffusion-v1-5/stable-diffusion-v1-5"
CLIP_MODEL_ID = "openai/clip-vit-base-patch16"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if Path("/content").exists():
    try:
        from google.colab import drive
        drive.mount("/content/drive")  # salvare imediată pe Drive, aceeași structură de directoare
    except Exception as exc:
        print("Google Drive nu a putut fi montat:", exc)
_DRIVE = Path("/content/drive/MyDrive")
if Path("/content").exists():
    OUTPUT_ROOT = (_DRIVE / "StableDiffusionLab5" / "outputs") if _DRIVE.exists() else Path("/content/StableDiffusionLab5/outputs")
else:
    OUTPUT_ROOT = Path.cwd() / "StableDiffusionLab5_outputs"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
# Un singur director per sesiune (la fiecare rulare a acestei celule); fiecare experiment are un subdirector numerotat.
SESSION_ID = f"session_{dt.datetime.now().astimezone().strftime('%Y%m%d_%H%M%S')}"
SESSION_ROOT = OUTPUT_ROOT / SESSION_ID
SESSION_ROOT.mkdir(parents=True, exist_ok=False)
print(f"Dispozitiv: {DEVICE}; director sesiune: {SESSION_ROOT}")
if DEVICE != "cuda": print("Atenție: laboratorul cere GPU CUDA; modelul nu va rula pe CPU.")

def json_safe(value):
    if isinstance(value, Path): return str(value)
    if isinstance(value, (np.integer,)): return int(value)
    if isinstance(value, (np.floating,)): return float(value)
    if isinstance(value, dict): return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)): return [json_safe(v) for v in value]
    return value

def save_json_atomic(path: Path, payload: dict):
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(json_safe(payload), ensure_ascii=False, indent=2), encoding="utf-8")
    temporary.replace(path)

def package_version(name):
    try: return version(name)
    except PackageNotFoundError: return "not-installed"

@dataclass
class RunPaths:
    root: Path
    samples: Path
    plots: Path
    exports: Path
    parameters: Path
    metrics: Path
    final_report: Path
    log: Path

def create_run_paths(configuration, name=None):
    now = dt.datetime.now().astimezone()
    import re as _re
    label = _re.sub(r"[^0-9A-Za-z]+", "_", str(name or configuration.get("experiment") or configuration.get("kind") or "run")).strip("_")[:60]
    index = sum(1 for p in SESSION_ROOT.iterdir() if p.is_dir()) + 1
    run_id = f"{index:02d}_{label}"
    root = SESSION_ROOT / run_id
    root.mkdir(parents=True, exist_ok=False)
    paths = RunPaths(root, root/'samples', root/'plots', root/'exports', root/'parameters.json', root/'metrics.json', root/'final_report.json', root/'execution.log')
    for folder in (paths.samples, paths.plots, paths.exports): folder.mkdir()
    manifest = {"started_at": now.isoformat(), "project": "Laboratorul 5 — Stable Diffusion", "run_id": run_id,
        "environment": {"python_version": platform.python_version(), "platform": platform.platform(), "device": DEVICE,
            "torch_cuda": torch.version.cuda, "library_versions": {n: package_version(n) for n in ("torch", "diffusers", "transformers", "accelerate", "torchmetrics")}},
        "reproducibility": {"seeds": ([item["seed"] for item in configuration["pairs"]] if configuration.get("pairs") is not None else configuration.get("seeds", [])), "deterministic_mode": False,
            "note": "Generator CUDA per imagine; determinism poate depinde de hardware, biblioteci și scheduler."},
        "configuration": json_safe(configuration)}
    save_json_atomic(paths.parameters, manifest)
    save_json_atomic(paths.metrics, {"run_directory": str(root), "metric_definitions": {"clip_score": "CLIPScore (100*max(cos,0), CLIPModel direct); mai mare indică potrivire text-imagine, nu calitate completă", "seconds_per_image": "timp CUDA sincronizat; prima generare este exclusă"}, "records": [], "summary": None})
    paths.log.write_text(f"{now.isoformat()} | INFO | Run started: {run_id}\n", encoding="utf-8")
    return paths

def log_message(paths, message, level="INFO"):
    line = f"{dt.datetime.now().astimezone().isoformat()} | {level} | {message}\n"
    with paths.log.open("a", encoding="utf-8") as stream: stream.write(line)
    print(line.rstrip())

## Persistența artefactelor

Fiecare experiment creează director propriu, cu identificator unic și fără suprascriere: parameters.json este scris înainte de generare; metrics.json este actualizat atomic; execution.log, samples/ (imagini și grilă), plots/ și exports/ (CSV-uri) rămân grupate per rulare. Nu este necesar checkpoint pentru inferență fără stare de antrenare. Rapoartele agregate primesc un director nou. În Colab fără Drive montat, /content este temporar; pentru persistență montați Drive și setați OUTPUT_ROOT către folderul montat.

In [ ]:
from diffusers import StableDiffusionPipeline, StableDiffusionImg2ImgPipeline, PNDMScheduler, DDIMScheduler, DPMSolverMultistepScheduler, EulerDiscreteScheduler
pipe, BASE_SCHEDULER_CONFIG, CLIP_METRIC = None, None, None
if RUN_EXPERIMENTS or RUN_IMG2IMG:
    if not torch.cuda.is_available(): raise RuntimeError("Este necesar un runtime GPU CUDA; nu se face fallback la CPU.")
    try:
        pipe = StableDiffusionPipeline.from_pretrained(MODEL_ID, torch_dtype=torch.float16, use_safetensors=True).to("cuda")
    except Exception as exc:
        raise RuntimeError(f"Încărcarea explicită {MODEL_ID!r} a eșuat. Verificați identificatorul, disponibilitatea, licența/condițiile Hugging Face și rețeaua. Nu s-a încercat fallback. Cauză: {type(exc).__name__}: {exc}") from exc
    BASE_SCHEDULER_CONFIG = dict(pipe.scheduler.config)
    print(f"Model: {MODEL_ID}; scheduler inițial: {pipe.scheduler.__class__.__name__}")
else:
    print("Modelul nu este descărcat. Setați RUN_EXPERIMENTS=True și reexecutați celula pentru încărcare.")

## Prompturi și configurația de bază

Prompturile din cerință sunt păstrate în engleză. Baza folosește 30 pași, guidance 7,5, 512×512, PNDM (scheduler-ul implicit SD 1.5) și seed-urile 42, 7, 123: 9 imagini. Prima imagine din fiecare apel este exclusă din media timpului (warm-up), dar rămâne în grilă și la evaluarea CLIP. Timpul este sincronizat CUDA înainte și după generare.

In [ ]:
PROMPTS = {"P1": "a watercolor painting of a red fox in a snowy forest", "P2": "a photo of a modern library interior, natural light", "P3": "a cartoon robot reading a book, flat illustration"}
BASE_CONFIG = {"experiment": "Bază", "prompt_keys": ["P1", "P2", "P3"], "seeds": [42, 7, 123], "pairs": None,
    "guidance_scale": 7.5, "num_inference_steps": 30, "scheduler": "pndm", "negative_prompt": None, "width": 512, "height": 512}
SCHEDULERS = {"pndm": PNDMScheduler, "ddim": DDIMScheduler, "dpm_solver_multistep": DPMSolverMultistepScheduler, "euler": EulerDiscreteScheduler}
ALL_RESULTS = []

class _ClipScorer:
    """CLIPScore = 100*max(cos(img,txt),0), calculat direct cu CLIPModel (independent de versiunea transformers/torchmetrics)."""
    def __init__(self, model_id):
        from transformers import CLIPModel, CLIPProcessor
        self.model = CLIPModel.from_pretrained(model_id).to("cuda").eval()
        self.processor = CLIPProcessor.from_pretrained(model_id)
    @torch.inference_mode()
    def score(self, images, prompts):
        inputs = self.processor(text=list(prompts), images=list(images), return_tensors="pt", padding=True, truncation=True, max_length=77).to("cuda")
        out = self.model(**inputs)
        cos = torch.nn.functional.cosine_similarity(out.image_embeds.float(), out.text_embeds.float(), dim=-1)
        return (100 * cos.clamp(min=0)).detach().cpu().tolist()

def _clip_metric():
    global CLIP_METRIC
    if CLIP_METRIC is None: CLIP_METRIC = _ClipScorer(CLIP_MODEL_ID)
    return CLIP_METRIC

def _image_tensor(image):
    return torch.from_numpy(np.asarray(image.convert("RGB"), dtype=np.uint8).copy()).permute(2, 0, 1).unsqueeze(0)

def _make_grid(images, labels, destination):
    cols, rows = min(3, max(1, len(images))), math.ceil(len(images) / min(3, max(1, len(images))))
    fig, axes = plt.subplots(rows, cols, figsize=(4*cols, 4*rows), squeeze=False)
    for ax in axes.flat: ax.axis("off")
    for ax, image, label in zip(axes.flat, images, labels): ax.imshow(image); ax.set_title(label, fontsize=9); ax.axis("off")
    fig.tight_layout(); fig.savefig(destination, dpi=140, bbox_inches="tight"); plt.show(); plt.close(fig)

def _pairs_for(config):
    pairs = config.get("pairs")
    if pairs is None: pairs = [{"prompt_key": k, "seed": s, "replicate": f"seed{s}"} for k in config["prompt_keys"] for s in config["seeds"]]
    if not pairs: raise ValueError("Configurația trebuie să conțină perechi prompt-seed.")
    for item in pairs:
        if item["prompt_key"] not in PROMPTS: raise ValueError(f"Prompt necunoscut: {item['prompt_key']}")
    return pairs

def _setting_label(cfg):
    return f"{cfg['num_inference_steps']} pași; guidance {cfg['guidance_scale']}; {cfg['scheduler']}; negative={cfg['negative_prompt'] is not None}"

def ruleaza(config):
    """Generează imaginile unei configurații, calculează CLIP și salvează progresul."""
    if pipe is None or DEVICE != "cuda": raise RuntimeError("Modelul nu este încărcat pe GPU. Activați GPU și rulați celula de încărcare.")
    cfg = copy.deepcopy(config)
    if cfg["scheduler"] not in SCHEDULERS: raise ValueError(f"Scheduler neacceptat: {cfg['scheduler']}")
    if cfg["width"] % 8 or cfg["height"] % 8: raise ValueError("Lățimea/înălțimea trebuie să fie multipli de 8.")
    pairs = _pairs_for(cfg)
    paths = create_run_paths(cfg, name=f"{cfg['experiment']}_{_setting_label(cfg)}"); log_message(paths, f"Configurație: {json.dumps(json_safe(cfg), ensure_ascii=False)}")
    images, labels, rows, prompts = [], [], [], []
    state = {"run_directory": str(paths.root), "metric_definitions": {"clip_score": "CLIPScore (100*max(cos,0)), mai mare = potrivire mai bună", "seconds_per_image": "secunde/image, CUDA sincronizat; warm-up exclus"}, "records": [], "summary": None}
    try:
        pipe.scheduler = SCHEDULERS[cfg["scheduler"]].from_config(BASE_SCHEDULER_CONFIG)
        metric = _clip_metric()
        for index, item in enumerate(pairs):
            key, seed = item["prompt_key"], int(item["seed"]); prompt = PROMPTS[key]
            generator = torch.Generator(device="cuda").manual_seed(seed)
            torch.cuda.synchronize(); started = time.perf_counter()
            image = pipe(prompt=prompt, negative_prompt=cfg["negative_prompt"], num_inference_steps=int(cfg["num_inference_steps"]),
                guidance_scale=float(cfg["guidance_scale"]), width=int(cfg["width"]), height=int(cfg["height"]), generator=generator).images[0]
            torch.cuda.synchronize(); elapsed = time.perf_counter() - started
            seconds = None if index == 0 else elapsed
            label = f"{key} | seed {seed} | {item.get('replicate', '')}".strip(" |")
            filename = f"{index+1:02d}_{key}_seed{seed}_{item.get('replicate', 'sample')}.png"
            image_path = paths.samples / filename; image.save(image_path)
            score = float(metric.score([image.convert("RGB")], [prompt])[0])
            images.append(image); labels.append(label); prompts.append(prompt)
            row = {"index": index, "prompt_key": key, "prompt": prompt, "seed": seed, "replicate": item.get("replicate", ""),
                "clip_score": score, "seconds_per_image": seconds, "image_path": str(image_path)}
            rows.append(row)
            state["records"].append({"step": index+1, "elapsed_seconds": seconds, "metrics": {"clip_score": score}, "status": "completed", "image": filename, "prompt_key": key, "seed": seed, "replicate": item.get("replicate", "")})
            save_json_atomic(paths.metrics, state)
            log_message(paths, f"Imagine {index+1}/{len(pairs)}; CLIP={score:.3f}; timp={seconds if seconds is not None else 'warm-up exclus'}")
        # Media se recalculează pe întregul lot (verificare independentă a scorurilor individuale).
        function_mean = float(np.mean(metric.score([im.convert("RGB") for im in images], prompts)))
        scores = [row["clip_score"] for row in rows]
        summary = {"experiment": cfg["experiment"], "setting": _setting_label(cfg), "n_images": len(rows), "clip_mean": function_mean,
            "clip_std": float(np.std(scores, ddof=1)) if len(scores)>1 else 0.0, "clip_individual_mean": float(np.mean(scores)),
            "clip_functional_difference": abs(function_mean-float(np.mean(scores))),
            "seconds_per_image_mean_excluding_warmup": float(np.mean([r["seconds_per_image"] for r in rows if r["seconds_per_image"] is not None])) if len(rows)>1 else None,
            "warmup_images_excluded_from_timing": 1, "run_directory": str(paths.root)}
        state["summary"] = summary; save_json_atomic(paths.metrics, state)
        _make_grid(images, labels, paths.samples / "grid.png"); _make_grid(images, labels, paths.plots / "image_grid.png")
        pd.DataFrame(rows).to_csv(paths.exports / "per_image.csv", index=False, encoding="utf-8-sig")
        pd.DataFrame([summary]).to_csv(paths.exports / "summary.csv", index=False, encoding="utf-8-sig")
        save_json_atomic(paths.final_report, {"selected_artifact": "samples/grid.png", "selection": {"criterion": "toate eșantioanele configurației", "direction": "not_applicable"}, "final_metrics": summary, "finished_at": dt.datetime.now().astimezone().isoformat(), "status": "completed"})
        log_message(paths, f"Run completed: {paths.root}")
        result = {"config": cfg, "summary": summary, "rows": rows, "images": images, "labels": labels, "paths": paths}; ALL_RESULTS.append(result)
        return result
    except Exception as exc:
        save_json_atomic(paths.metrics, state)
        save_json_atomic(paths.final_report, {"status": "failed", "error": f"{type(exc).__name__}: {exc}", "finished_at": dt.datetime.now().astimezone().isoformat()})
        log_message(paths, f"Run failed: {type(exc).__name__}: {exc}", "ERROR"); raise

def _show_summary(results):
    frame = pd.DataFrame([r["summary"] for r in results])
    display(frame[["experiment", "setting", "n_images", "clip_mean", "clip_std", "seconds_per_image_mean_excluding_warmup"]])

## Imaginile de bază (3 prompturi × 3 seed-uri)

Rulați baseline-ul înainte de comparații. Fiecare rerulare primește director nou. Scorul CLIP (100·max(cos,0)) este calculat direct cu CLIPModel openai/clip-vit-base-patch16, individual și pe lot. Abaterea standard este pe scorurile per imagine (ddof=1).

In [ ]:
BASE_RESULT = None
if RUN_EXPERIMENTS:
    BASE_RESULT = ruleaza(copy.deepcopy(BASE_CONFIG)); _show_summary([BASE_RESULT])
else:
    print("Sărit: setați RUN_EXPERIMENTS=True și reexecutați configurarea și încărcarea pentru a genera imaginile.")

## Experimente A–E

Fiecare configurație pornește de la BASE_CONFIG și schimbă un singur parametru, în afara designului factorial C: în C scheduler-ele se compară la același număr de pași (10 sau 30), iar 10 vs. 30 se compară separat în interiorul scheduler-ului. Astfel, fiecare comparație controlată de scheduler menține constante celelalte condiții. Seed-urile folosesc un generator CUDA explicit.

- **A — guidance:** 1, 3, 7,5, 12, 20.
- **B — pași:** 5, 10, 20, 50.
- **C — scheduler:** DDIM, DPM-Solver multistep și Euler, la 10 și 30 pași.
- **D — seed:** P1 cu seed-urile 42, 7, 123, 2025 și încă o generare cu seed 42.
- **E — prompt negativ:** absent sau blurry, low quality, text, watermark.

Nu declarați o diferență concludentă dacă diferența mediilor CLIP este mai mică decât abaterea standard observată.

In [ ]:
A_RESULTS, B_RESULTS, C_RESULTS, E_RESULTS, D_RESULT = [], [], [], [], None
if RUN_EXPERIMENTS:
    for value in [1, 3, 7.5, 12, 20]:
        cfg = copy.deepcopy(BASE_CONFIG); cfg["experiment"] = "A — guidance"; cfg["guidance_scale"] = value
        A_RESULTS.append(ruleaza(cfg))
    _show_summary(A_RESULTS)
else: print("Sărit: RUN_EXPERIMENTS=False")

In [ ]:
if RUN_EXPERIMENTS:
    for value in [5, 10, 20, 50]:
        cfg = copy.deepcopy(BASE_CONFIG); cfg["experiment"] = "B — pași"; cfg["num_inference_steps"] = value
        B_RESULTS.append(ruleaza(cfg))
    _show_summary(B_RESULTS)
else: print("Sărit: RUN_EXPERIMENTS=False")

In [ ]:
if RUN_EXPERIMENTS:
    for steps in [10, 30]:
        for scheduler_name in ["ddim", "dpm_solver_multistep", "euler"]:
            cfg = copy.deepcopy(BASE_CONFIG); cfg["experiment"] = "C — scheduler"
            cfg["num_inference_steps"] = steps; cfg["scheduler"] = scheduler_name
            C_RESULTS.append(ruleaza(cfg))
    _show_summary(C_RESULTS)
else: print("Sărit: RUN_EXPERIMENTS=False")

In [ ]:
if RUN_EXPERIMENTS:
    cfg = copy.deepcopy(BASE_CONFIG); cfg["experiment"] = "D — seed"; cfg["prompt_keys"] = ["P1"]
    cfg["pairs"] = [{"prompt_key":"P1", "seed":42, "replicate":"seed42_original"},
        {"prompt_key":"P1", "seed":7, "replicate":"seed7"}, {"prompt_key":"P1", "seed":123, "replicate":"seed123"},
        {"prompt_key":"P1", "seed":2025, "replicate":"seed2025"}, {"prompt_key":"P1", "seed":42, "replicate":"seed42_repeat"}]
    D_RESULT = ruleaza(cfg)
    first = np.asarray(D_RESULT["images"][0], dtype=np.int16); repeated = np.asarray(D_RESULT["images"][4], dtype=np.int16)
    diff = float(np.abs(first-repeated).mean())
    comparison = {"seed_first":42, "seed_repeat":42, "mean_absolute_pixel_difference_0_255":diff,
        "mean_absolute_pixel_difference_0_1":diff/255.0, "exact_pixel_match":bool(np.array_equal(first,repeated)),
        "note":"Diferențe nenule pot apărea din cauza nondeterminismului hardware/bibliotecilor."}
    save_json_atomic(D_RESULT["paths"].exports / "seed_repeat_comparison.json", comparison)
    saved = json.loads(D_RESULT["paths"].metrics.read_text(encoding="utf-8")); saved["seed_repeat_comparison"] = comparison
    save_json_atomic(D_RESULT["paths"].metrics, saved); print(json.dumps(comparison, ensure_ascii=False, indent=2))
else: print("Sărit: RUN_EXPERIMENTS=False")

In [ ]:
if RUN_EXPERIMENTS:
    for negative in [None, "blurry, low quality, text, watermark"]:
        cfg = copy.deepcopy(BASE_CONFIG); cfg["experiment"] = "E — prompt negativ"; cfg["negative_prompt"] = negative
        E_RESULTS.append(ruleaza(cfg))
    _show_summary(E_RESULTS)
else: print("Sărit: RUN_EXPERIMENTS=False")

## Tabel și grafice cu bare de eroare (din rulări reale)

Codul citește numai metrics.json din rulări completed; nu inventează valori lipsă. Generează tabelul CSV (câte un rând/rulare) și graficele A și B (media ± abaterea standard CLIP). Raportul PDF final este generat de celula strictă de la sfârșitul notebook-ului.

In [ ]:
def load_completed_run_summaries():
    found = []
    for metrics_path in SESSION_ROOT.glob("*/metrics.json"):
        try: payload = json.loads(metrics_path.read_text(encoding="utf-8"))
        except (OSError, json.JSONDecodeError): continue
        summary = payload.get("summary")
        if not summary or "experiment" not in summary or "setting" not in summary: continue
        try: final = json.loads((metrics_path.parent / "final_report.json").read_text(encoding="utf-8"))
        except (OSError, json.JSONDecodeError): continue
        if final.get("status") == "completed": found.append(summary)
    return found

def _plot_parameter(frame, prefix, x_column, title, destination):
    subset = frame[frame["experiment"].str.startswith(prefix)].copy()
    if subset.empty: return False
    subset = subset.sort_values(x_column)
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.errorbar(subset[x_column], subset["clip_mean"], yerr=subset["clip_std"], fmt="o-", capsize=5)
    ax.set_xlabel(x_column.replace("_", " ")); ax.set_ylabel("Scor CLIP mediu ± abatere standard")
    ax.set_title(title); ax.grid(True, alpha=0.3); fig.tight_layout(); fig.savefig(destination, dpi=160)
    plt.show(); plt.close(fig); return True

def export_table_and_plots():
    summaries = load_completed_run_summaries()
    if not summaries: raise RuntimeError("Nu există rulări completed; rulați baseline-ul sau un experiment înainte de export.")
    frame = pd.DataFrame(summaries); pass
    a_mask, b_mask = frame["experiment"].str.startswith("A"), frame["experiment"].str.startswith("B")
    frame["guidance"] = np.nan; frame["steps"] = np.nan
    if a_mask.any(): frame.loc[a_mask, "guidance"] = frame.loc[a_mask, "setting"].str.extract(r"guidance ([0-9.]+)")[0].astype(float)
    if b_mask.any(): frame.loc[b_mask, "steps"] = frame.loc[b_mask, "setting"].str.extract(r"([0-9]+) pași")[0].astype(float)
    paths = create_run_paths({"kind":"aggregate_report", "source":"metrics.json reale din run-uri completed", "run_count":len(frame)}, name="tabel_agregat")
    frame.drop(columns=["guidance", "steps"], errors="ignore").to_csv(paths.exports / "results_table.csv", index=False, encoding="utf-8-sig")
    if a_mask.any(): _plot_parameter(frame[a_mask], "A", "guidance", "Experimentul A — guidance vs scor CLIP", paths.plots / "experiment_A_guidance.png")
    if b_mask.any(): _plot_parameter(frame[b_mask], "B", "steps", "Experimentul B — pași vs scor CLIP", paths.plots / "experiment_B_steps.png")
    save_json_atomic(paths.metrics, {"run_directory":str(paths.root), "metric_definitions":{"source":"rulări completed persistate"}, "records":[], "summary":{"source_runs":len(frame)}})
    save_json_atomic(paths.final_report, {"selected_artifact":"exports/results_table.csv", "selection":{"criterion":"tabel agregat din toate rulările completed", "direction":"not_applicable"}, "final_metrics":{"source_runs":len(frame)}, "finished_at":dt.datetime.now().astimezone().isoformat(), "status":"completed"})
    log_message(paths, f"Tabel și grafice exportate din {len(frame)} rulări")
    display(frame[["experiment","setting","n_images","clip_mean","clip_std","seconds_per_image_mean_excluding_warmup"]])
    print(f"Tabel: {paths.exports / 'results_table.csv'}")
    return paths

if load_completed_run_summaries(): TABLE_PATHS = export_table_and_plots()
else: print("Tabelul se generează după prima rulare finalizată; nu există valori fabricate.")

## Recomandări și întrebări de control — completați după rulare

Propuneți două configurații în 5–10 rânduri: (1) calitate maximă, timpul nefiind prioritar; (2) viteză, de exemplu un serviciu cu țintă sub 2 s. Verificați ținta pe hardware-ul măsurat; nu presupuneți că este atinsă. Susțineți recomandările cu CLIP ± abatere, timp și inspecție vizuală. Dacă diferența CLIP este mai mică decât variabilitatea observată, numiți comparația neconcludentă.

Răspundeți la: 
1) Ce se întâmplă la guidance 1 și 20 și de ce? 
2) De la câți pași nu mai vedeți îmbunătățiri clare; coincide cu CLIP? 
3) De ce DPM-Solver/Euler se descurcă mai bine la puțini pași decât DDPM din Laboratorul 4? 
4) De ce același seed dă aceeași imagine și la ce folosește? 
5) Ce nu măsoară CLIP; dați exemplu din imagini. 
6) Ce riscuri etice sau juridice există într-o companie?



## Opțional (+1): img2img (dezactivat implicit)

Folosește o imagine generată anterior și același prompt/seed, variind numai strength (0,3; 0,6; 0,9). Strength mai mare înseamnă mai multă corupere înainte de denoising și, de regulă, mai multă libertate față de imaginea sursă. Necesită runtime GPU și modelul încărcat.

In [ ]:
RUN_IMG2IMG = True
def ruleaza_img2img(source_image, prompt, strength, seed=42):
    if pipe is None or DEVICE != "cuda": raise RuntimeError("Încărcați modelul pe GPU înainte de img2img.")
    cfg = {"experiment":"Bonus img2img", "strength":float(strength), "seed":int(seed), "prompt":prompt,
        "width":source_image.width, "height":source_image.height, "guidance_scale":7.5}
    paths = create_run_paths(cfg, name=f"img2img_strength_{strength}_seed_{seed}"); log_message(paths, f"Img2img: strength={strength}, seed={seed}")
    img_pipe = StableDiffusionImg2ImgPipeline(**pipe.components).to("cuda")
    generator = torch.Generator(device="cuda").manual_seed(int(seed))
    torch.cuda.synchronize(); started = time.perf_counter()
    output = img_pipe(prompt=prompt, image=source_image.convert("RGB"), strength=float(strength), guidance_scale=7.5,
        num_inference_steps=30, generator=generator).images[0]
    torch.cuda.synchronize(); elapsed = time.perf_counter()-started
    output_path = paths.samples / "img2img.png"; output.save(output_path)
    score = float(_clip_metric().score([output.convert("RGB")], [prompt])[0])
    summary = {"experiment":"Bonus img2img", "setting":f"strength={strength}", "n_images":1,
        "clip_mean":score, "clip_std":0.0, "seconds_per_image_mean_excluding_warmup":elapsed, "run_directory":str(paths.root)}
    save_json_atomic(paths.metrics, {"run_directory":str(paths.root), "metric_definitions":{"clip_score":"CLIPScore (100*max(cos,0))"},
        "records":[{"step":1,"elapsed_seconds":elapsed,"metrics":{"clip_score":score},"status":"completed"}], "summary":summary})
    pd.DataFrame([summary]).to_csv(paths.exports / "summary.csv", index=False, encoding="utf-8-sig")
    _make_grid([output], [f"strength={strength}"], paths.plots / "image_grid.png")
    save_json_atomic(paths.final_report, {"selected_artifact":"samples/img2img.png", "final_metrics":summary,
        "finished_at":dt.datetime.now().astimezone().isoformat(), "status":"completed"})
    log_message(paths, f"Img2img completed: {paths.root}")
    return {"image":output, "summary":summary, "paths":paths}

if RUN_IMG2IMG:
    if BASE_RESULT is None: raise RuntimeError("Rulați întâi baseline-ul pentru a selecta o imagine sursă.")
    source = BASE_RESULT["images"][0]
    IMG2IMG_RESULTS = [ruleaza_img2img(source, PROMPTS["P1"], value, seed=42) for value in (0.3, 0.6, 0.9)]
else: print("Bonus img2img dezactivat; activați RUN_IMG2IMG=True pentru a-l rula.")

## Raport PDF final (strict)

Completați răspunsurile din `REPORT_TEXT` (recomandările se calculează automat), apoi rulați celula de export. Raportul (4 pagini: tabel, grila de bază + grafice A/B, C/D/E, text) este scris în `exports/lab5_report.pdf` al unui subdirector nou al sesiunii, fără suprascriere. Exportul eșuează cu listă de probleme dacă lipsește vreun rezultat al experimentelor, vreo metrică, rularea D sau vreun răspuns, și nu creează niciun PDF în acel caz. Notă: la C, variantele de 10 pași schimbă și numărul de pași (controlul corespunzător este B la 10 pași, scheduler PNDM).

In [ ]:
# Completați din tabelele voastre. Câmpurile goale blochează exportul PDF.
REPORT_TEXT = {
    "answers": {
        1: "La 1, scorul scade clar și variază mult de la un prompt la altul. Guidance-ul clasifier-free combină predicția fără prompt cu cea cu prompt: eps = eps_necond + g·(eps_cond − eps_necond). La g=1 se folosește doar predicția condiționată, fără întărirea promptului. Imaginile devin vagi sau ignoră o parte din prompt. La 20, predicția este împinsă prea departe de cea necondiționată. Tipic apar culori suprasaturate, contrast exagerat și artefacte, adică imagini „arse”. CLIP nu penalizează valoarea 20, fiindcă măsoară potrivirea cu textul, nu aspectul. Din cifre, g=3–20 e un platou, iar diferențele sunt sub abaterea standard.", 
        2: "De la aproximativ 20 de pași nu mai apar îmbunătățiri clare: peste 20, diferențele de 0,3–0,7 sunt mult sub abaterea standard de ~2. Costul în timp continuă să crească liniar, deci peste 20 plătești timp fără câștig. Cu doar 9 imagini, nu pot spune sigur că 20 și 50 sunt egali, doar că nu se vede o diferență. Verifică și vizual detaliile fine, pe care CLIP nu le vede bine. La 5 pași abaterea standard ajunge la 7,2, deci unele imagini eșuează complet.", 
        3: "La 10 pași: DPM-Solver 34,9, DDIM 34,2, Euler 34,2, față de PNDM 31,6. DDPM din Lab 4 face pași stocastici mici. Aproximarea gaussiană a pasului invers este corectă doar dacă pasul este mic, adică ~1000 de pași. Cu 10 pași ar fi puternic degradat. DDIM, Euler și DPM-Solver rezolvă ecuația diferențială deterministă asociată modelului (probability-flow ODE). Traiectoria este netedă, deci pași mari rămân precişi. DPM-Solver adaugă un solver de ordin superior care exploatează structura semi-liniară a ecuației. Comparația corectă cu Lab 4 este DDIM-ul de acolo, care a dat rezultate bune și la puțini pași (FID 43,6 la 50 de pași și 45,7 la 20). Aici lucrăm în plus într-un spațiu latent, cu un model mult mai mare.", 
        4: "Seedul fixează zgomotul latent inițial. Dacă restul parametrilor sunt la fel (model, prompt, scheduler, pași, guidance), procesul este determinist. Experimentul D (5 repetări cu același seed) a avut abatere standard 0,39, nu 0. Diferența probabil vine din nedeterminismul numeric pe GPU (fp16, kernel-uri atomice). Nu am verificat pixel cu pixel dacă imaginile sunt identice. Folosește la: comparații controlate (se schimbă un singur factor), reproductibilitate, depanare și partajarea unei rețete exacte.",  
        5: "CLIP măsoară doar similaritatea semantică globală dintre imagine și text.", 
        6: "Drepturi de autor și date de antrenare, Proprietate intelectuală asupra rezultatelor, Licență, Date personale și chipuri (GDPR), Mărci comerciale și logo-uri"
        },
}

In [ ]:
# Export strict al raportului PDF (4 pagini) în exports/ al unui subdirector nou al sesiunii.
# Folosește exclusiv rezultatele din această sesiune (BASE_RESULT, A_RESULTS, B_RESULTS, C_RESULTS, D_RESULT, E_RESULTS).
# Dacă lipsește orice rezultat sau text cerut, ridică o eroare și NU creează niciun PDF/director.
import textwrap
from matplotlib.backends.backend_pdf import PdfPages

MAX_TEXT_LINES = 60
NEG = "blurry, low quality, text, watermark"

def export_final_report_pdf():
    problems = []
    g = globals()
    base, D = g.get("BASE_RESULT"), g.get("D_RESULT")
    A, B, C, E = (g.get(n) or [] for n in ("A_RESULTS", "B_RESULTS", "C_RESULTS", "E_RESULTS"))

    def check(result, label, allowed, n_images):
        if not isinstance(result, dict) or "summary" not in result:
            problems.append(f"{label}: rezultat lipsă"); return
        s = result["summary"]
        if s.get("n_images") != n_images: problems.append(f"{label}: {s.get('n_images')} imagini, se cer {n_images}")
        for key in ("clip_mean", "clip_std", "seconds_per_image_mean_excluding_warmup"):
            v = s.get(key)
            if v is None or not math.isfinite(v): problems.append(f"{label}: metrica {key} lipsește sau nu este finită")
        changed = {k for k in BASE_CONFIG if k != "experiment" and result["config"].get(k) != BASE_CONFIG[k]}
        if not changed <= allowed: problems.append(f"{label}: parametri schimbați față de bază: {sorted(changed - allowed)}")
        try:
            status = json.loads(result["paths"].final_report.read_text(encoding="utf-8")).get("status")
        except (OSError, json.JSONDecodeError):
            status = None
        if status != "completed": problems.append(f"{label}: final_report.json lipsește sau nu are status completed")

    def by(results, getter):
        return {getter(r["config"]): r for r in results if isinstance(r, dict) and "config" in r}

    check(base, "Bază", set(), 9)
    a_map = by(A, lambda c: c["guidance_scale"]); b_map = by(B, lambda c: c["num_inference_steps"])
    c_map = by(C, lambda c: (c["scheduler"], c["num_inference_steps"]))
    e_map = by(E, lambda c: c["negative_prompt"])
    expected_c = {(s, n) for s in ("ddim", "dpm_solver_multistep", "euler") for n in (10, 30)}
    for name, got, want in (("A guidance", set(a_map), {1, 3, 7.5, 12, 20}), ("B pași", set(b_map), {5, 10, 20, 50}),
                            ("C scheduler", set(c_map), expected_c), ("E prompt negativ", set(e_map), {None, NEG})):
        if got != want: problems.append(f"{name}: lipsă {sorted(map(str, want - got))}, neașteptat {sorted(map(str, got - want))}")
    for v, r in a_map.items(): check(r, f"A {v}", {"guidance_scale"}, 9)
    for v, r in b_map.items(): check(r, f"B {v}", {"num_inference_steps"}, 9)
    for v, r in c_map.items(): check(r, f"C {v}", {"scheduler", "num_inference_steps"}, 9)
    for v, r in e_map.items(): check(r, f"E {v}", {"negative_prompt"}, 9)
    check(D, "D", {"prompt_keys", "pairs"}, 5)
    comparison = None
    if isinstance(D, dict) and "paths" in D:
        try: comparison = json.loads((D["paths"].exports / "seed_repeat_comparison.json").read_text(encoding="utf-8"))
        except (OSError, json.JSONDecodeError): problems.append("D: lipsește seed_repeat_comparison.json (rulați celula D)")
    rt = g.get("REPORT_TEXT") or {}
    texts = {f"răspuns {k}": v for k, v in rt.get("answers", {}).items()}
    if len(texts) != 6: problems.append("REPORT_TEXT: structură incompletă (se cer 6 răspunsuri)")
    for k, v in texts.items():
        if not isinstance(v, str) or not v.strip(): problems.append(f"REPORT_TEXT: câmpul '{k}' este gol")
    if problems:
        raise RuntimeError("Export PDF refuzat; lipsesc rezultate/texte reale:\n- " + "\n- ".join(problems))

    def wrapped(title, body):
        return [title] + textwrap.wrap(body.strip(), 170, initial_indent="    ", subsequent_indent="    ")
    cand = [(e, v, r["summary"]) for e, v, r in ([("Bază", "30 pași, ghidare 7,5, PNDM", base)]
            + [("A", f"ghidare {k}", a_map[k]) for k in a_map] + [("B", f"{k} pași", b_map[k]) for k in b_map]
            + [("C", f"{k[0]}, {k[1]} pași", c_map[k]) for k in c_map] + [("E", "fără prompt negativ" if k is None else "cu prompt negativ", e_map[k]) for k in e_map])]
    ssec = lambda s: s["seconds_per_image_mean_excluding_warmup"]
    fmt = lambda c: f"{c[0]} ({c[1]}): CLIP {c[2]['clip_mean']:.2f} ± {c[2]['clip_std']:.2f}, {ssec(c[2]):.2f} s/imagine"
    best = max(cand, key=lambda c: c[2]["clip_mean"])
    fast_ok = [c for c in cand if ssec(c[2]) < 2.0]
    fast = max(fast_ok, key=lambda c: c[2]["clip_mean"]) if fast_ok else min(cand, key=lambda c: ssec(c[2]))
    note = lambda c: ("Diferența față de Bază este sub abaterea standard, deci comparația este neconcludentă. "
                      if abs(c[2]["clip_mean"] - base["summary"]["clip_mean"]) < max(c[2]["clip_std"], base["summary"]["clip_std"]) else "")
    lines = ["RECOMANDĂRI (calculate automat din rezultate)"]
    lines += wrapped("Calitate maximă:", f"cel mai mare CLIP mediu: {fmt(best)}. {note(best)}Verificați vizual detaliile pe care CLIP nu le măsoară.")
    lines += wrapped("Viteză:", (f"cel mai bun CLIP sub 2 s/imagine: {fmt(fast)}. " if fast_ok else f"nicio configurație sub 2 s; cea mai rapidă: {fmt(fast)}. ") + note(fast))
    lines += ["", "ÎNTREBĂRI DE CONTROL"]
    for i in range(1, 7): lines += wrapped(f"{i}.", rt["answers"][i])
    if len(lines) > MAX_TEXT_LINES:
        raise RuntimeError(f"Textul raportului are {len(lines)} linii (maxim {MAX_TEXT_LINES} pentru 4 pagini); scurtați-l.")

    rows = [("Bază", "30 pași, ghidare 7,5, PNDM", base)]
    rows += [("A", f"ghidare {v}", a_map[v]) for v in sorted(a_map)] + [("B", f"{v} pași", b_map[v]) for v in sorted(b_map)]
    rows += [("C", f"{s}, {n} pași", c_map[(s, n)]) for (s, n) in sorted(c_map)]
    rows += [("D", "P1, seed 42/7/123/2025/42 repetat", D)]
    rows += [("E", "fără prompt negativ" if k is None else "cu prompt negativ", e_map[k]) for k in (None, NEG)]
    sources = [r["summary"]["run_directory"] for _, _, r in rows]
    table = pd.DataFrame([{"experiment": e, "value": v, "n_images": r["summary"]["n_images"], "clip_mean": r["summary"]["clip_mean"],
        "clip_std": r["summary"]["clip_std"], "seconds_per_image": r["summary"]["seconds_per_image_mean_excluding_warmup"],
        "run_directory": r["summary"]["run_directory"]} for e, v, r in rows])

    paths = create_run_paths({"kind": "final_report", "source_runs": sources}, name="raport_final")
    figures = []
    try:
        log_message(paths, f"Construire raport final din {len(rows)} rulări reale")
        table.to_csv(paths.exports / "final_results_table.csv", index=False, encoding="utf-8-sig")

        fig = plt.figure(figsize=(11.7, 8.3)); ax = fig.add_subplot(111); ax.axis("off")
        ax.text(0, 1.0, "Laboratorul 5 — Stable Diffusion: raport din rulări reale", fontsize=16, va="top")
        ax.text(0, 0.95, f"Model {MODEL_ID}; CLIP {CLIP_MODEL_ID}; 512×512; seed-uri 42/7/123; generat {dt.datetime.now().astimezone().isoformat(timespec='seconds')}", fontsize=8, va="top")
        shown = table[["experiment", "value", "n_images", "clip_mean", "clip_std", "seconds_per_image"]].copy()
        shown[["clip_mean", "clip_std", "seconds_per_image"]] = shown[["clip_mean", "clip_std", "seconds_per_image"]].round(3)
        t = ax.table(cellText=shown.values, colLabels=["Experiment", "Valoare", "N", "CLIP mediu", "Abatere std", "Timp/imagine (s)"], loc="center", cellLoc="left", bbox=[0, 0.05, 1, 0.84])
        t.auto_set_font_size(False); t.set_fontsize(8)
        figures.append(fig)

        fig = plt.figure(figsize=(11.7, 8.3)); gs = fig.add_gridspec(2, 2, height_ratios=[1.1, 1])
        ax0 = fig.add_subplot(gs[0, :]); ax0.imshow(Image.open(base["paths"].samples / "grid.png")); ax0.axis("off"); ax0.set_title("Grila de bază (3 prompturi × 3 seed-uri)")
        for col, (name, mapping, xlabel) in enumerate((("A — guidance", a_map, "guidance_scale"), ("B — pași", b_map, "num_inference_steps"))):
            ax = fig.add_subplot(gs[1, col]); xs = sorted(mapping)
            ax.errorbar(xs, [mapping[x]["summary"]["clip_mean"] for x in xs], yerr=[mapping[x]["summary"]["clip_std"] for x in xs], fmt="o-", capsize=5)
            ax.set_title(f"Experimentul {name}"); ax.set_xlabel(xlabel); ax.set_ylabel("Scor CLIP mediu ± abatere std"); ax.grid(alpha=0.3)
        fig.tight_layout(); figures.append(fig)

        fig = plt.figure(figsize=(11.7, 8.3)); gs = fig.add_gridspec(2, 3, height_ratios=[1, 1.1])
        ax = fig.add_subplot(gs[0, 0]); keys = sorted(c_map)
        ax.bar(range(len(keys)), [c_map[k]["summary"]["clip_mean"] for k in keys], yerr=[c_map[k]["summary"]["clip_std"] for k in keys], capsize=4)
        ax.set_xticks(range(len(keys))); ax.set_xticklabels([f"{s[:4]}/{n}" for s, n in keys], rotation=45, ha="right"); ax.set_title("C — scheduler/pași"); ax.set_ylabel("Scor CLIP mediu ± std")
        ax = fig.add_subplot(gs[0, 1]); d_rows = D["rows"]
        ax.bar(range(len(d_rows)), [r["clip_score"] for r in d_rows]); ax.set_xticks(range(len(d_rows))); ax.set_xticklabels([r["replicate"] for r in d_rows], rotation=45, ha="right")
        ax.set_title("D — CLIP per seed (P1)"); ax.set_ylabel("Scor CLIP")
        ax.text(0.5, 0.97, f"seed 42 vs repetat: |Δpixel| medie = {comparison['mean_absolute_pixel_difference_0_255']:.4f}/255", transform=ax.transAxes, ha="center", va="top", fontsize=7)
        ax = fig.add_subplot(gs[0, 2])
        ax.bar(["fără", "cu"], [e_map[k]["summary"]["clip_mean"] for k in (None, NEG)], yerr=[e_map[k]["summary"]["clip_std"] for k in (None, NEG)], capsize=4)
        ax.set_title("E — prompt negativ"); ax.set_ylabel("Scor CLIP mediu ± std")
        ax = fig.add_subplot(gs[1, :]); ax.imshow(Image.open(D["paths"].samples / "grid.png")); ax.axis("off"); ax.set_title("D — grila seed-urilor (P1)")
        fig.tight_layout(); figures.append(fig)

        fig = plt.figure(figsize=(11.7, 8.3)); fig.text(0.03, 0.97, "\n".join(lines), va="top", fontsize=7.5, linespacing=1.15)
        figures.append(fig)

        pdf_path = paths.exports / "lab5_report.pdf"; tmp_path = pdf_path.with_suffix(".pdf.tmp")
        with PdfPages(tmp_path) as pdf:
            for f in figures: pdf.savefig(f)
            page_count = pdf.get_pagecount()
        if not 2 <= page_count <= 4 or tmp_path.stat().st_size == 0: raise RuntimeError(f"PDF invalid: {page_count} pagini")
        tmp_path.replace(pdf_path)
        for f in figures: plt.close(f)
        save_json_atomic(paths.metrics, {"run_directory": str(paths.root), "metric_definitions": {"source": "rezumate din rulări completed"}, "records": [], "summary": {"source_runs": sources, "pages": page_count}})
        save_json_atomic(paths.final_report, {"selected_artifact": "exports/lab5_report.pdf", "selection": {"criterion": "raport final din rulările experimentale ale sesiunii", "direction": "not_applicable"}, "final_metrics": {"pages": page_count, "source_runs": len(sources)}, "finished_at": dt.datetime.now().astimezone().isoformat(), "status": "completed"})
        log_message(paths, f"Raport PDF salvat: {pdf_path}")
        print(f"PDF: {pdf_path}\nDirector rulare: {paths.root}")
        return pdf_path
    except Exception as exc:
        for f in figures: plt.close(f)
        save_json_atomic(paths.final_report, {"status": "failed", "error": f"{type(exc).__name__}: {exc}", "finished_at": dt.datetime.now().astimezone().isoformat()})
        log_message(paths, f"Export eșuat: {type(exc).__name__}: {exc}", "ERROR"); raise

FINAL_REPORT_PDF = export_final_report_pdf()


## Salvare locală

Artefactele sunt scrise imediat în `SESSION_ROOT` (sub `OUTPUT_ROOT`, pe Drive, dacă este montat). Celula următoare arhivează directorul sesiunii (`session_.../<experiment>/...`, structura păstrată) și descarcă arhiva pe PC.

In [ ]:

import shutil, zipfile

def export_run_zip(root, include_checkpoints=False, max_checkpoint_mb=None, download=True):
    """Arhivează directorul rulării păstrând structura și, în Colab, îl descarcă pe PC-ul local."""
    root = Path(root).resolve(); base = root.parent
    out = base / (root.name + ("_complet" if include_checkpoints else "") + ".zip")
    with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as zf:
        for f in sorted(root.rglob("*")):
            if not f.is_file() or f.suffix == ".tmp": continue
            if not include_checkpoints and f.suffix in (".pt", ".pth", ".ckpt", ".safetensors"): continue
            zf.write(f, f.relative_to(base).as_posix())
    print(f"Arhivă: {out} ({out.stat().st_size/1e6:.1f} MB)")
    if download and Path("/content").exists():
        from google.colab import files
        files.download(str(out))
    return out

export_run_zip(SESSION_ROOT)
